In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import copy


train = pd.read_excel("BNT-Energy2f.xlsx")
X=train.iloc[:,:-1]
Y=train.iloc[:,-1]

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.3, random_state=42)


scalerX = StandardScaler().fit(X_train)
scalery = StandardScaler().fit(y_train.values.reshape(-1,1))
X_train1 = scalerX.transform(X_train)
y_train1 = scalery.transform(y_train.values.reshape(-1,1))
X_test1 = scalerX.transform(X_test)
y_test1 = scalery.transform(y_test.values.reshape(-1,1))
y_new_inverse = scalery.inverse_transform(y_test1.reshape(-1,1))
X_train2=copy.deepcopy(X_train1)
y_train2=copy.deepcopy(y_train1)


from sklearn.linear_model import ElasticNet, Lasso, LinearRegression
from xgboost import XGBRegressor
# from sklearn.base import BaseEstimator, TransformerMixin, RegressorMixin, clone
from sklearn.model_selection import KFold, cross_val_score
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error,r2_score,mean_squared_error

xgb_model=XGBRegressor()

search_space={'n_estimators':[100,200,300,400,500,600,800],'max_depth':[3,4,6,7,9],'gamma0':[0.01],'learning_rate':[0.001,0.01,0.05,0.1,0.5,1]}


from sklearn.model_selection import GridSearchCV

GS=GridSearchCV(estimator=xgb_model,param_grid=search_space,scoring=["r2","neg_root_mean_squared_error"],refit="neg_root_mean_squared_error",cv=5,verbose=4)
GS.fit(X_train1,y_train1)

print(GS.best_params_)



In [ ]:
model_xgb=XGBRegressor(n_estimators=600,max_depth=4,eta=0.1)
model_xgb.fit(X_train1,y_train1)

In [ ]:
import numpy as np
from xgboost import XGBRegressor
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import pandas as pd


xBi = np.linspace(0.1,0.6,51) # zero beginning is not acceptable in few elements
xNa = np.linspace(0.1,0.6,51)
xSr = np.linspace(0,0.2,21)   # keep electric field in the reasonable range for better output
xCa = np.linspace(0,0.2,21)   # try Zr, Nb and Mg in place of Hf as well
xTi = np.linspace(0.5,1,51)
xBa = np.linspace(0,0.3,31)


lists=[]

for i in range(len(xBi)):
  print(i)
  for j in range(len(xNa)):
    for k in range(len(xSr)):
      for l in range(len(xCa)):
        for m in range(len(xTi)):
          for n in range(len(xBa)):
            if(xBi[i]+xNa[j]+xSr[k]+xBa[n]+xCa[l]==1):
              if(xTi[m]==1):
                if(3*xBi[i]+1*xNa[j]+2*xSr[k]+2*xBa[n]+2*xCa[l]==2):
                  if(4*xTi[m]==4):
                    Bi = xBi[i]
                    Na = xNa[j]
                    Sr = xSr[k]
                    Ca = xCa[l]
                    Ti = xTi[m]
                    Ba = xBa[n]

                    PA = Bi*7.4+Na*24.11+Sr*27.6+Ba*39.7+Ca*22.8
                    PB = Ti*14.6
                    ENA = Bi*2.14+Na*0.89+Sr*1.13+Ba*1.08+Ca*1.17
                    ENB = Ti*1.86
                    VA = Bi*21.31+Na*23.78+Sr*33.94+Ba*38.16+Ca*26.2
                    VB = Ti*10.64
                    ATWT = Bi*208.98+Na*22.98+Sr*87.9+Ti*47.95+Ba*137.9+Ca*39.96
                    ATNO = Bi*83+Na*11+Sr*38+Ti*22+Ba*56+Ca*20
                    IEA = Bi*7.28+Na*5.14+Sr*5.69+Ba*5.21+Ca*6.11
                    IEB = Ti*6.83
                    CRSA = Bi*1.997+Na*2.65+Sr*3.21+Ba*3.402+Ca*3
                    CRSB = Ti*2.58
                    Eb = 200
                    K = 0
                    Rb = 0
                    Eu = 0
                    La = 0
                    Zr = 0
                    Sn = 0
                    Ta = 0
                    Nd = 0
                    Al = 0
                    Nb = 0
                    Mg = 0
                    Hf = 0
                    Li = 0
                    Fe = 0
                    Ag = 0
                    Sc = 0
                    Sb = 0
                    Sm = 0

                    vector=[Bi,Na,K,Ca,Sr,Rb,Eu,Ba,La,Ti,Zr,Sn,Ta,Nd,Al,Nb,Mg,Hf,Li,Fe,Ag,Sc,Sb,Sm,PA,PB,ENA,ENB,VA,VB,CRSA,CRSB,IEA,IEB,ATWT,ATNO,Eb]
                    vector_t=scalerX.transform(np.asarray(vector).reshape(1,-1))
                    [output]=model_xgb.predict(vector_t)
                    vector.append(scalery.inverse_transform(output.reshape(-1,1)))
                    lists.append(vector)

print(lists)
df = pd.DataFrame(lists)

# Save the DataFrame to an Excel file
df.to_excel('reverse-engg-WrecSrCaBa-200.xlsx', index=False)

In [ ]:
import numpy as np
xBi = np.linspace(0.0,0.1,11)
print(xBi)

[0.   0.01 0.02 0.03 0.04 0.05 0.06 0.07 0.08 0.09 0.1 ]


In [ ]:
RA = 0.5*1.35+0.35*1.39+0.15*1.64                     # Ionic Radii: K 1.64 Na1.39 Bi1.35 Hf 0.71 Ti 0.61 O 1.40 Bi0.5Na0.35K0.15Ti0.9Hf0.1O3
RB = 0.9*0.61+0.1*0.71
RO = 1.40
TF = (RA+RO)/(1.414*(RB+RO))  #Tolerance Factor Calculation 0.8 ≤ t ≤ 1.0 → stable perovskite
print(TF)

OctaF= RB/RO  # 0.414 ≤ μ ≤ 0.732 stable perovskit

print(OctaF)


0.9829218423964038
